## Reasoning Patch

In [20]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Overview

### Set-up

In [28]:
import torch
import gc
import random
import pandas as pd
import re
from tqdm import tqdm

import sys
sys.path.append("src")
import _util
import _dataset
import _prompt
import _mapping
from _intervention import prepare_batch_multitoken_intervention, batch_intervene, get_attention_freeze_hooks

In [22]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A5500
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |   4245 MiB |  10132 MiB | 293648 MiB | 289403 MiB |
|       from large pool |   4244 MiB |  10131 MiB | 293647 MiB | 289403 MiB |
|       from small pool |      1 MiB |      1 MiB |      1 MiB |      0 MiB |
|---------------------------------------------------------------------------|
| Active memory         |   4245 MiB |  10132 MiB | 293648 MiB | 289403 M

In [23]:
model_type = "GPT-OSS" # GPT-OSS or R1

if model_type == "GPT-OSS":
    model, tokenizer = _util.load_OSS()
elif model_type == "R1":
    model, tokenizer = _util.load_R1()

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [24]:
random.seed(42)

add_ds = _dataset.create_h_dataset(num_digits=3, num_samples=276)


In [25]:
divide_num = 26

attention_prompts = []
for add_ds_entry in add_ds[-20:]:
    base_prompt = _prompt.get_stepwise_prompt(add_ds_entry["base_1_digits"], add_ds_entry["base_2_digits"], add_ds_entry["base_1_num"], add_ds_entry["base_2_num"])
    truncated_base_prompt = _prompt.divide_prompt(divide_num, base_prompt)[0]
    
    attention_prompts.append(truncated_base_prompt)

print(attention_prompts)


['<|start|>system<|message|>You are ChatGPT, a large language model trained by OpenAI.\nKnowledge cutoff: 2024-06\nCurrent date: 2025-06-28\n\nReasoning: high\n\n# Valid channels: analysis, commentary, final. Channel must be included for every message.<|end|><|start|>developer<|message|># Instruction: In analysis, add two numbers stepwise. In final, output only the sum.<|end|><|start|>user<|message|>What is 104+112?<|end|><|start|>assistant<|channel|>analysis<|message|>The user asks: "What is 104+112?" The instruction from the developer says: "In the analysis, add two numbers stepwise. For the final output, output only the sum." So we need to do the addition stepwise in the analysis, and then in the final output, just output the sum." So I need to do stepwise addition in analysis, then output only the sum in final. So in analysis, I will show the stepwise addition: 104 + 112. Let\'s do it: 104 + 112 = 104 + 100 + 10 + 2 = 104 + 100 = 204, 204 + 10 = 214, 214 + 2 = 216. So the sum is ',

In [26]:
attention_tokens = tokenizer(attention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
attention_freeze_hooks = get_attention_freeze_hooks(model, attention_tokens)

In [27]:
prompt_type = "h_pre_result" # empty or pre_result or pre_final_sum or h or h_pre_result or h_pre_final_sum
if prompt_type:
    prompt_type = "_" + prompt_type

# Load the divided prompts dataset
if 'h1' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h1_prompts{prompt_type[3:]}.csv")
elif 'h' in prompt_type:
    prompts = pd.read_csv(f"data/{model_type}/h_prompts{prompt_type[2:]}.csv")
else:
    prompts = pd.read_csv(f"data/{model_type}/prompts{prompt_type}.csv")
prompts["base_sum"] = prompts["base_sum"].astype('Int64')
prompts["source_sum"] = prompts["source_sum"].astype('Int64')
print(f"loaded {len(prompts)} prompts")

loaded 256 prompts


In [29]:
intervention_loc = [25] # restatement or reasoning or restatement_and_reasoning

if type(intervention_loc) == str:
    if 'h' in prompt_type:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit_h
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit_h
    else:
        if model_type == "GPT-OSS":
            intervention_ids_dict = _mapping.intervene_ids_stepwise_3_digit
        elif model_type == "R1":
            intervention_ids_dict = _mapping.intervene_ids_R1_3_digit

    if intervention_loc == "restatement":
        intervention_ids = intervention_ids_dict["restatement"]
    elif intervention_loc == "reasoning":
        intervention_ids = intervention_ids_dict["reasoning"]
    elif intervention_loc == "restatement_and_reasoning":
        intervention_ids = intervention_ids_dict["restatement"] + intervention_ids_dict["reasoning"]
elif type(intervention_loc) == list:
    intervention_ids = intervention_loc

print(intervention_ids)

[25]


In [30]:
intervention_id_to_tok_pos = {
    6: 93,
    8: 111,
    10: 201,
    12: 211,
    14: 217,
    18: 229,
    19: 232,
    20: 235,
    22: 241,
    23: 244,
    25: 250,
    26: 257,
    27: 266,
}

tok_pos_list = [intervention_id_to_tok_pos[id] for id in intervention_ids]
print(tok_pos_list)


[250]


## Frozen Attention Patching

In [ ]:
LAYER = 0

header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _util.create_csv_file(f"experiments/activation_intervention/output/{model_type}/frozen_attention", f"{prompt_type[1:]}_final_sum.csv", header, overwrite=True)

batch_size = 1

for i, row in tqdm(prompts.iterrows(), total=len(prompts)):

    base_prompt = row['base_prompt']
    base_prompts = [row['base_prompt']] * len(attention_prompts)
    base_labels_str = [str(row['base_sum'])] * len(attention_prompts)
    base_labels = tokenizer(base_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)

    source_prompt = row['source_prompt']
    source_prompts = [row['source_prompt']] * len(attention_prompts)
    source_labels_str = [str(row['source_sum'])] * len(attention_prompts)
    source_labels = tokenizer(source_labels_str, add_special_tokens=False, return_tensors="pt")["input_ids"].squeeze(1)

    tokens, source_tokens, hook = prepare_batch_multitoken_intervention(model, tokenizer, LAYER, tok_pos_list, base_prompts, source_prompts)
    input_length = tokens["input_ids"].shape[1]

    with torch.no_grad():
        output = batch_intervene(model, tokens["input_ids"], attention_freeze_hooks + [hook], attention_mask=tokens["attention_mask"])
    pred_toks = output.logits[:,-1,:].argmax(dim=-1)
    prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
    base_prob = prob[torch.arange(prob.shape[0]), base_labels]
    source_prob = prob[torch.arange(prob.shape[0]), source_labels]
    tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
    del output
    
    for j in range(len(attention_prompts)):
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _util.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), generated_text, base_prob[j].item(), source_prob[j].item()])

    del tokens, hook
    torch.cuda.empty_cache()
    gc.collect()


100%|█████████████████████████████████████████| 256/256 [40:20<00:00,  9.45s/it]
